# Notebook 26 — Universality Test of the Effective Noise Coordinate

This notebook tests whether the effective noise description found earlier is specific to one optimized protocol point, or whether it persists across a small family of nearby protocols.

Core question:

**Does the same kind of reduced noise geometry survive when we vary the coherent control parameters?**

We test several protocol variants near the shaped phase-locked baseline and ask:

1. what best-fit effective direction
   `gamma_eff = gamma + lambda * gamma_phi`
   each variant prefers,
2. whether the fitted `lambda` stays stable,
3. whether the learned 1D response curves remain similar,
4. whether one shared effective-coordinate picture continues to organize the noisy gate-quality map.

This is the natural next step after Notebook 25 because it distinguishes:
- a one-off fit,
- from a more universal effective-noise structure.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Baseline protocol and nearby variants

In [ ]:
baseline = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

protocols = {
    'baseline': dict(baseline),
    'T_plus': {**baseline, 'T': 28.0},
    'T_minus': {**baseline, 'T': 24.0},
    'alpha_plus': {**baseline, 'alpha': 0.12},
    'alpha_minus': {**baseline, 'alpha': 0.08},
    'omega_plus': {**baseline, 'omega_max': 1.15},
    'omega_minus': {**baseline, 'omega_max': 1.03},
}

for name, pset in protocols.items():
    print(name, pset)


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, params, gamma_decay=0.0, gamma_phi=0.0, n_steps=180):
    H = build_time_dependent_hamiltonian(
        params['T'], params['omega_max'], params['alpha'], V, params['delta0'], params['p'], params['q']
    )
    times = np.linspace(0.0, params['T'], n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            amp = basis_state.overlap(state)
            vals.append(amp)
        else:
            val = (basis_state.dag() * state * basis_state)
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(params, gamma_decay=0.0, gamma_phi=0.0, n_steps=180):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            amp = basis_state.overlap(final_state)
            score = np.abs(amp) ** 2
        else:
            val = (basis_state.dag() * final_state * basis_state)
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Noise grid

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 21)
gamma_phi_vals = np.linspace(0.0, 0.12, 21)


## Build balanced order-parameter maps for all variants

In [ ]:
variant_results = {}

for name, params in protocols.items():
    cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            U_eff, finals = build_noisy_effective_map(
                params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=180
            )
            cz_map[i, j] = compensated_cz_fidelity(U_eff)
            coh_map[i, j] = coherence_proxy(finals)
            leak_map[i, j] = leakage_from_finals(finals)

    S = cz_map * coh_map * (1.0 - leak_map)
    S_norm = S / np.max(S) if np.max(S) > 0 else S.copy()

    variant_results[name] = {
        'params': params,
        'cz_map': cz_map,
        'coh_map': coh_map,
        'leak_map': leak_map,
        'S_norm': S_norm,
    }
    print("done:", name)


## Fit lambda for each variant

In [ ]:
def fit_lambda_for_map(S_norm):
    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

    def loss(lam):
        gamma_eff_phi = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(gamma_eff_phi, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method='bounded')
    return float(fit.x), float(fit.fun), S_gamma, S_phi

for name, data in variant_results.items():
    lam, loss, S_gamma, S_phi = fit_lambda_for_map(data['S_norm'])
    data['lambda'] = lam
    data['axis_loss'] = loss
    data['S_gamma'] = S_gamma
    data['S_phi'] = S_phi
    print(name, "lambda =", lam, "axis loss =", loss)


## Compare fitted lambdas and collapse losses

In [ ]:
names = list(variant_results.keys())
lambdas = [variant_results[name]['lambda'] for name in names]
losses = [variant_results[name]['axis_loss'] for name in names]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].bar(names, lambdas)
axes[0].set_ylabel('best-fit lambda')
axes[0].set_title('Effective noise direction across protocol variants')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, losses)
axes[1].set_ylabel('axis-slice MSE')
axes[1].set_title('Collapse loss across protocol variants')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## Build 1D response curves for all variants

In [ ]:
for name, data in variant_results.items():
    lam = data['lambda']
    S_norm = data['S_norm']

    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    gamma_eff_flat = gamma_eff_grid.ravel()
    S_flat = S_norm.ravel()

    order = np.argsort(gamma_eff_flat)
    gamma_eff_sorted = gamma_eff_flat[order]
    S_sorted = S_flat[order]

    n_bins = 20
    bins = np.linspace(gamma_eff_sorted.min(), gamma_eff_sorted.max(), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(n_bins, np.nan)

    for k in range(n_bins):
        mask = (gamma_eff_sorted >= bins[k]) & (gamma_eff_sorted < bins[k+1])
        if np.any(mask):
            means[k] = np.mean(S_sorted[mask])

    valid = ~np.isnan(means)
    data['gamma_eff_centers'] = centers[valid]
    data['response_curve'] = means[valid]


## Compare effective-response curves

In [ ]:
plt.figure(figsize=(7.6, 5.2))
for name in names:
    plt.plot(
        variant_results[name]['gamma_eff_centers'],
        variant_results[name]['response_curve'],
        label=name
    )

plt.xlabel('gamma_eff')
plt.ylabel('Normalized balanced order parameter')
plt.title('Response curves across nearby protocol variants')
plt.legend()
plt.tight_layout()
plt.show()


## Sensitivity comparison

In [ ]:
sensitivity_summary = {}

for name, data in variant_results.items():
    x = data['gamma_eff_centers']
    y = data['response_curve']
    dy = np.gradient(y, x)
    d2y = np.gradient(dy, x)

    idx_drop = int(np.argmin(dy))
    idx_curv = int(np.argmin(d2y))

    data['dy'] = dy
    data['d2y'] = d2y
    data['peak_drop_x'] = float(x[idx_drop])
    data['peak_drop_val'] = float(dy[idx_drop])
    data['peak_curv_x'] = float(x[idx_curv])
    data['peak_curv_val'] = float(d2y[idx_curv])

    sensitivity_summary[name] = (
        data['peak_drop_x'],
        data['peak_curv_x']
    )

    print(name,
          "fastest drop at", data['peak_drop_x'],
          "strongest curvature at", data['peak_curv_x'])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for name in names:
    axes[0].plot(
        variant_results[name]['gamma_eff_centers'],
        variant_results[name]['dy'],
        label=name
    )
    axes[1].plot(
        variant_results[name]['gamma_eff_centers'],
        variant_results[name]['d2y'],
        label=name
    )

axes[0].set_xlabel('gamma_eff')
axes[0].set_ylabel('dS / dgamma_eff')
axes[0].set_title('Sensitivity across variants')
axes[0].legend()

axes[1].set_xlabel('gamma_eff')
axes[1].set_ylabel('d²S / dgamma_eff²')
axes[1].set_title('Curvature across variants')
axes[1].legend()

plt.tight_layout()
plt.show()


## Show a representative family of normalized maps

In [ ]:
show_names = ['baseline', 'T_plus', 'alpha_plus', 'omega_plus']
fig, axes = plt.subplots(1, len(show_names), figsize=(16, 4.2))

for ax, name in zip(axes, show_names):
    arr = variant_results[name]['S_norm']
    im = ax.imshow(
        arr,
        origin='lower',
        aspect='auto',
        extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
        vmin=0, vmax=1
    )
    ax.set_title(name)
    ax.set_xlabel('gamma')
    ax.set_ylabel('gamma_phi')

fig.colorbar(im, ax=axes.tolist(), label='normalized S')
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Universality test of the effective noise coordinate")
lines.append("")
lines.append("Protocol variants tested near the phase-locked baseline:")
for name in names:
    pset = variant_results[name]['params']
    lines.append(
        f"- {name}: T={pset['T']:.2f}, alpha={pset['alpha']:.2f}, "
        f"Omega={pset['omega_max']:.2f}, Delta0={pset['delta0']:.2f}"
    )

lines.append("")
lines.append("Best-fit effective directions:")
for name in names:
    lines.append(
        f"- {name}: lambda={variant_results[name]['lambda']:.4f}, "
        f"axis-loss={variant_results[name]['axis_loss']:.6e}, "
        f"fastest-drop gamma_eff={variant_results[name]['peak_drop_x']:.4f}"
    )

lambda_vals = np.array([variant_results[name]['lambda'] for name in names])
lines.append("")
lines.append(f"Lambda mean ± std = {lambda_vals.mean():.4f} ± {lambda_vals.std():.4f}")
lines.append("")
lines.append("Interpretation:")
lines.append("- If lambda stays in a narrow range, the effective direction is structurally stable.")
lines.append("- If the response curves remain similar, the reduced noise geometry is shared across nearby protocols.")
lines.append("- If the rapid-degradation band stays near the same gamma_eff, the threshold picture is not a one-off artifact.")

summary_text = "\n".join(lines)
print(summary_text)


## Final conclusion

This notebook tests whether the effective-noise picture is universal within a local family of nearby shaped protocols.

The strongest possible outcome is:

- similar fitted `lambda`,
- similar 1D response curves,
- similar rapid-degradation bands.

If that happens, then the effective coordinate is not just a fit to one optimized point. It becomes a structural feature of the local protocol family.

That gives the cleanest next-level statement:

**the noisy phase-locked CZ regime is governed by a locally universal effective noise geometry rather than by unrelated noise responses at each individual protocol point.**
